<a href="https://colab.research.google.com/github/Shanmukh-dev/Custom-GPT/blob/main/RAG_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import tiktoken
import torch
from torch import nn
import datasets


tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
engine_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")

with open("model.py", "w") as f:
    f.write(engine_res.text)
    # print(engine_res.text)
print("Downloaded model.py")

# Downloading train_utils.py
train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")

with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
    # print(engine_res.text)
print("Downloaded train_utils.py")

data_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py")

with open("data.py", "w") as f:
    f.write(data_res.text)
    # print(engine_res.text)
print("Downloaded data.py")

In [ ]:
from model import *
from train_utils import *
from data import *


config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=4,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

In [ ]:
# import pandas as pd

# data = pd.read_csv("timekeepers_sft_200k.csv")
# df = pd.DataFrame(data)
# # df.head(5)
# eval(df["conversation_messages_json"][0])


import datasets

grounded_rag = datasets.load_dataset("shanaka95/GroundedRAG")
print(len(grounded_rag))

In [ ]:
def process_data_in_chunks(dataset, tokenizer, chunk_size=20000, max_samples=400000):
    all_tokens = []
    current_chunk_text = ""
    count = 0

    for sample in dataset:
        query = f"<user>{sample['question']}</user>"
        context = f"<context>{sample['context']}</context>"
        response = f"<assistant>{sample['answer']}</assistant>"
        current_chunk_text += f"{query}\n{context}\n{response}\n"
        count += 1

        if count % chunk_size == 0:
            # Tokenize the accumulated chunk of text and clear memory
            chunk_tokens = tokenizer.encode(current_chunk_text, allowed_special="all")
            all_tokens.extend(chunk_tokens)
            current_chunk_text = ""
            print(f"Processed {count} samples...")

        if count >= max_samples:
            break

    # Process any remaining text
    if current_chunk_text:
        chunk_tokens = tokenizer.encode(current_chunk_text, allowed_special="all")
        all_tokens.extend(chunk_tokens)

    # Append EOF token
    eof_tokens = tokenizer.encode("<|endoftext|>", allowed_special="all")
    all_tokens.extend(eof_tokens)

    return all_tokens

In [ ]:
from tqdm.auto import tqdm
import numpy as np

def create_store_dataset():
    print("Tokenizing dataset in chunks...")
    tokens_list = process_data_in_chunks(grounded_rag["train"], tokenizer, chunk_size=25000, max_samples=100000)

    print("Converting tokens to NumPy array...")
    tokens = np.asarray(tokens_list, dtype=np.uint16)

    print(f"Saving {len(tokens)} tokens to sft.npy...")
    mmap = np.lib.format.open_memmap("sft.npy", dtype=tokens.dtype, mode="w+", shape=tokens.shape)
    mmap[:] = tokens
    mmap.flush()
    print("Successfully saved tokens.")

create_store_dataset()

In [ ]:
from torch.utils.data import Dataset, DataLoader
class MidTrainDataset(Dataset):
    def __init__(self, path, block_size):
        self.data = np.load(path, mmap_mode="r", allow_pickle=True)
        self.block_size = block_size
        self.n_samples = (len(self.data)-1) // self.block_size

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        start = idx * self.block_size
        end = start + self.block_size
        x = self.data[start:end]
        y = self.data[start+1:end+1]

        return (
            torch.from_numpy(x.astype(np.int64)),
            torch.from_numpy(y.astype(np.int64))
        )


In [ ]:
full_dataset = MidTrainDataset("sft.npy", config.block_size)

train_split = int(0.9*len(full_dataset))

train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_split))
test_dataset = torch.utils.data.Subset(full_dataset, range(train_split, len(full_dataset)))

train_dataloader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True)

print(len(full_dataset))
print(len(train_dataset))
print(len(test_dataset))

In [ ]:
model = CustomGPT(config)
state_dict = torch.load("tara_n1_30m_it_v1.pth", map_location=device)
state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()
calc_params(model)

In [ ]:
from tqdm.auto import tqdm

steps = 10000
test_count = 1000
test_steps = 1000

train_iter = iter(train_dataloader)

avg_train_loss = 0

for step in tqdm(range(1, steps+1)):
    model.train()
    if not train_iter:
        train_iter = iter(train_dataloader)

    train_loss = train_step(model, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps=1)
    avg_train_loss += train_loss
    if step%test_count == 0:
        avg_train_loss /= test_count

        test_loss = test_step(model, test_dataloader, test_steps, loss_fn, device)

        print(f"Step: {step} | Train loss: {avg_train_loss} | Test loss: {test_loss}")
        avg_train_loss = 0

In [ ]:
prompt = "How much is ATLAS?"
context = torch.tensor(tokenizer.encode(f"<user>{prompt}</user>\n<context>[Document1]This is a lusury watch company that makes custom watches as well. our falgship models are ATLAS - $3000; SOLSTICE - $4000; EXPEDITION - $1500</context>\n<assistant>"), dtype=torch.long, device=device).unsqueeze(0)

with torch.inference_mode():
    generated = model.generate(context, max_new_tokens=512)

result = tokenizer.decode(generated[0].tolist())

print(result)

In [ ]:
def run_chat(model, tokenizer, device, max_new_tokens=256):
    # Simple array for chat memory
    chat_history = []

    print("Chat interface initialized. Type 'exit' or 'quit' to end the session.\n")

    while True:
        user_input = input("User: ")
        if user_input.strip().lower() in ['exit', 'quit']:
            print("Ending chat session.")
            break

        # Add user message to history
        chat_history.append({"role": "user", "content": user_input})

        # 1. Format the ENTIRE conversational history
        formatted_prompt = ""
        for msg in chat_history:
            if msg["role"] == "user":
                formatted_prompt += f"<user>{msg['content']}</user>\n"
            else:
                formatted_prompt += f"<assistant>{msg['content']}</assistant>\n"

        # Append the assistant tag to prompt the model to generate the next response
        formatted_prompt += "<assistant>"

        # 2. Tokenize the ENTIRE conversational context
        context = torch.tensor(tokenizer.encode(formatted_prompt, allowed_special="all"), dtype=torch.long, device=device).unsqueeze(0)

        model.eval()
        with torch.inference_mode():
            # Pass the entire conversational token history to the model
            generated = model.generate(context, max_new_tokens=max_new_tokens)

        # Decode results
        full_generation = tokenizer.decode(generated[0].tolist())

        # Extract only the newly generated response for this turn
        new_response = full_generation[len(formatted_prompt):]
        if "</assistant>" in new_response:
            new_response = new_response.split("</assistant>")[0].strip()
        else:
            new_response = new_response.strip()

        print(f"Assistant: {new_response}\n")

        # Add assistant response to history so it persists in the next turn's tokenization
        chat_history.append({"role": "assistant", "content": new_response})

# Run the updated chat interface
run_chat(model, tokenizer, device, 512)

In [ ]:
torch.save(model.state_dict(), "tara_n1_rag_v1.pth")